In [ ]:
%load_ext autoreload
%autoreload 1

%aimport bialignment
import bialignment as ba
import timeit

In [ ]:
inputfiles = ['Examples/DNAPolymerase1_Ecoli_AS-Struktur.txt',
         'Examples/DNAPolymerase1_Xanthomonas_AS-Struktur.txt'
        ]
input = [ ba.read_molecule_from_file(f) for f in inputfiles ]

# optionally, truncate input
for x in input:
    for i in range(2):
        x[i] = x[i][:160] # define how to truncate here

print(input)

args = {'type': 'Protein',
        'gap_cost': -50,
        'gap_opening_cost': -200,
        'shift_cost': -210,
        'structure_weight': 800,
        'max_shift': 1,
        'simmatrix': 'BLOSUM62',
        'nameA': 'Ecoli',
        'nameB': 'Xanthomonas',
        'nodescription': False,
        'outmode': 'full'
       }

In [ ]:
remake = False #True
%store -r
try:
    print(stored_alilines.keys())
except:
    stored_alilines = dict()

for ms in range(3):
    if not remake and (f'max_shift {ms}') in stored_alilines:
        continue
        
    args["max_shift"] = ms

    bialigner = ba.BiAligner(input[0][0],input[1][0],
                             input[0][1],input[1][1], 
                             **args)

    score = timeit.timeit(lambda:bialigner.optimize(),number=1)
    print(score)
    als = list(bialigner.decode_trace_full())
    for i,line in enumerate(als):
        print(f"{i:2} {line[0]:12} {line[1]}")

    stored_alilines[(f'max_shift {ms}')] = als
%store stored_alilines

In [ ]:
#computation time: max_shift 0: 0.4 min  (26 s)
print(f"{25.8/60:.2f} min")
#computation time: max_shift 1: 9.4 min
print(f"{566.0/60:.2f} min")
#computation time: max_shift 2: 32.2 min
print(f"{1930.9/60:.2f} min")

In [ ]:
alilines = stored_alilines['max_shift 2']

aliblocks = ba.breaklines(alilines, 80)
for block in aliblocks:
    for i,(name,aliline) in enumerate(block):
        print(f"{i:2} {name:18} {aliline}")
    print()

In [ ]:
for s in range(3):
    alilines = stored_alilines[f'max_shift {s}']
    ba.plot_alignment(alilines, 120, outname=f"dnapoly1-ms{s}-sc-210-sw800.svg")

# Example of Figure 1

In [ ]:
nameA = 'A'
nameB = 'B'
strA = "CHHHHHHHHHHHHHCCCCTCEEEEEEECCTCEEEEEEEECCC"
seqA = "RAKLPLKEKKLTATANYHPGIRYIMTGYSAKYIYSSTYARFR"
seqB = "KAKLPLKEKKLTRTANYHPGIRYIMTGYSAKRIYSSTYAYFR"
strB = "HHHHHHHHHHHHCCCCCCTCEEEEEEECCCCCEEEEEEEECC"

ba.plot_alignment([(nameA, seqA), (nameB, seqB), ('',strA), ('',strB)], 80,
    name_offset=3, show_position_numbers=False, outname = "fig1A.svg")

In [ ]:
seqA1 = "RAKLPLKEKKLTATANYH-PGIRYIMTGYSAK-YIYSSTYARFR"
strA1 = "CHHHHHHHHHHHHHCCCC-TCEEEEEEECCTC-EEEEEEEECCC"
strB1 = "-HHHHHHHHHHHHCCCCCCTCEEEEEEECCCCCEEEEEEEECC-"
seqB1 = "-KAKLPLKEKKLTRTANYHPGIRYIMTGYSAKRIYSSTYAYFR-"

ba.plot_alignment([(nameA, seqA1), (nameB, seqB1), ('',strA1), ('',strB1)], 80,
    name_offset=3, show_position_numbers=False, outname = "fig1B.svg")

In [ ]:
args['nameA'] = 'A'
args['nameB'] = 'B'
args['max_shift'] = 1
args['shift_cost'] = -150
args['structure_weight'] = 800

print(args)
print()

bialigner = ba.BiAligner(seqA, seqB, strA, strB,
                         **args)

score = bialigner.optimize()
print('SCORE',score)
print()
alilines = list(bialigner.decode_trace_full())
for i,line in enumerate(alilines):
    print(f"{i:2} {line[0]:18} {line[1]}")

In [ ]:
ba.plot_alignment(alilines, 80, show_position_numbers=False,
    name_offset=3, outname = "fig1-shift.svg")

In [ ]:
print(f'strA2 = "{alilines[6][1]}"')
print(f'seqA2 = "{alilines[1][1]}"')
print(f'seqB2 = "{alilines[3][1]}"')
print(f'strB2 = "{alilines[8][1]}"')
print(f'shtA2 = "{alilines[12][1]}"')
print(f'shtB2 = "{alilines[13][1]}"')

In [ ]:
strA2 = "CHHHHHHHHHHHHHCCCC-TCEEEEEEECCTC-EEEEEEEECCC"
seqA2 = "-RAKLPLKEKKLTATANY-HPGIRYIMTGYSAKYIYSSTYARFR"
seqB2 = "-KAKLPLKEKKLTRTANY-HPGIRYIMTGYSAKRIYSSTYAYFR"
strB2 = "-HHHHHHHHHHHHCCCCCCTCEEEEEEECCCCCEEEEEEEECC-"
shtA2 = ">...............................<..........."
shtB2 = "..................>........................<"

ba.plot_alignment([(nameA, seqA2), (nameB, seqB2), ('',strA2), ('',strB2), ('',shtA2), ('',shtB2)], 80,
    name_offset=3, show_position_numbers=False, outname = "fig1B.svg")

In [ ]:
strA2 = "CHHHHHHHHHHHHH-CCCCTCEEEEEEECCTC-EEEEEEEECCC"
seqA2 = "-RAKLPLKEKKLTATANYHPGIRYIMTGYSAKYIYSSTYAR-FR"
seqB2 = "-KAKLPLKEKKLTRTANYHPGIRYIMTGYSAKRIYSSTYAY-FR"
strB2 = "-HHHHHHHHHHHHCCCCCCTCEEEEEEECCCCCEEEEEEEE-CC"
shtA2 = ">.............<.................<........>.."
shtB2 = "............................................"

ba.plot_alignment([(nameA, seqA2), (nameB, seqB2), ('',strA2), ('',strB2), ('',shtA2), ('',shtB2)], 80,
    name_offset=3, show_position_numbers=False, outname = "fig1B.svg")

In [ ]:
strA2 = "CHHHHHHHHHHHHH-CCCCTCEEEEEEECCTC-EEEEEEEECCC"
seqA2 = "RAKLPLKEKKLTAT-ANYHPGIRYIMTGYSAKYIYSSTYAR-FR"
seqB2 = "KAKLPLKEKKLTRT-ANYHPGIRYIMTGYSAKRIYSSTYAY-FR"
strB2 = "-HHHHHHHHHHHHCCCCCCTCEEEEEEECCCCCEEEEEEEE-CC"
shtA2 = "................................<........>.."
shtB2 = "<.............>............................."

ba.plot_alignment([(nameA, seqA2), (nameB, seqB2), ('',strA2), ('',strB2), ('',shtA2), ('',shtB2)], 80,
    name_offset=3, show_position_numbers=False, outname = "fig1B.svg")

In [ ]:
nameA = 'A'
nameB = 'B'
strA = "CHHHHHHHHHHHHHCCCCTCEEEEEEECCTCEEEEEEEECCC"
seqA = "RAKLPLKEKKLTATANYHPGIRYIMTGYSAKYIYSSTYARFR"
seqB = "KAKLPLKEKKLTRTANYHPGIRYIMTGYSAKRIYSSTYAYFR"
strB = "HHHHHHHHHHHHCCCCCCTCEEEEEEECCCCCEEEEEEEECC"

ba.plot_alignment([(nameA, seqA), (nameB, seqB), ('',strA), ('',strB)], 80,
    name_offset=3, show_position_numbers=False, outname = "fig1A.svg")